# Flight Data
### Data Engineering Capstone Project

#### Project Summary
In this project, flight and airport data is gathered, transformed and stored for further analysis.

The project follows the follow steps:
* Step 1: Scope the Project and Gather Data
* Step 2: Explore and Assess the Data
* Step 3: Define the Data Model
* Step 4: Run ETL to Model the Data
* Step 5: Complete Project Write Up

In [1]:
# Do all imports and installs here
import os
import configparser
import requests
from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, udf, col, year, month, dayofmonth, hour, isnan, when, count
from pyspark.sql.types import TimestampType, FloatType

### Step 1: Scope the Project and Gather Data

#### Scope 
I am using airport data and actual flight traffic data. The data could be used to analyze how much traffic there is at different airports in different countries and when most traffic occurs. I will use Spark for most of my work but also some other Python libraries like datetime, requests, pathlib and configparser.

#### Describe and Gather Data 
The airport data is provided with the project. It includes mostly data to identify the airport and data about its location. Flight traffic data is retrieved from the [FlightAware API](https://flightaware.com/commercial/flightxml/). This flight data includes information about the aircraft used, its origin and its destination. For more detail see the inspections in the code below.

The flight aware API is a commercial API. You need an API key to send requests. Since there are costs associated with each request, I will store the gathered traffic data in directories in the project directory. The code to retrieve flight data from the API is included for reference only. It will only work if there are valid credentials in the file credentials.cfg.

In [2]:
# read credentials for FlightAware API

config = configparser.ConfigParser()
config.read('credentials.cfg')

os.environ['AWS_ACCESS_KEY_ID'] = config['AWS']['AWS_ACCESS_KEY_ID']
os.environ['AWS_SECRET_ACCESS_KEY'] = config['AWS']['AWS_SECRET_ACCESS_KEY']

fxml_user = config['FXML']['FXML_USERNAME']
fxml_api_key = config['FXML']['FXML_API_KEY']

In [3]:
# initialize Spark

spark = SparkSession \
    .builder \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:2.7.0") \
    .getOrCreate()

In [4]:
def set_max_result_size(max_size):
    """
    Set the maximum number of results in the response of the FlightAware API.
    """
    fxml_url = "https://flightxml.flightaware.com/json/FlightXML2/"

    payload = {'max_size':str(max_size)}
    response = requests.get(fxml_url + "SetMaximumResultSize",
        params=payload, auth=(fxml_user, fxml_api_key))

    if response.status_code == 200:
        print("Successfully set max size")
    else:
        print("Error executing request")

In [5]:
def get_for_airport(operation, airport, idx):
    """
    Get departures from, arrivals to and flights enroute to an airport using the FlightAware API.
    Results are stored to different directories for arrivals, departures and flights enroute.
    A seperate JSON-File is created for each airport.
    An index is appended to the file to distinguish results gathered at different times.
    """
    fxml_url = "https://flightxml.flightaware.com/json/FlightXML2/"

    airport = airport
    filename = "{}/{}-{}.json".format(operation, airport, idx)

    payload = {'airport':airport, 'howMany':'15000'}
    response = requests.get(fxml_url + operation,
        params=payload, auth=(fxml_user, fxml_api_key))

    if response.status_code == 200:
        f = open(filename, "w")
        f.write(response.text)
        f.close()
    else:
        print("Error executing request")

In [6]:
def get_all_airports():
    """
    Retrieve all airport codes known to the FlightXML API
    """
    fxml_url = "https://flightxml.flightaware.com/json/FlightXML2/"

    filename = "all_airports.txt"

    response = requests.get(fxml_url + "AllAirports",
        auth=(fxml_user, fxml_api_key))

    if response.status_code == 200:
        f = open(filename, "w")
        f.write(response.text)
        f.close()
    else:
        print("Error executing request")

In [7]:
# Load the given airport codes

ac_df = spark.read.csv("airport-codes_csv.csv", header=True)

In [8]:
# inspect the airport code data

print(ac_df.count())
ac_df.printSchema()

55075
root
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- elevation_ft: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- coordinates: string (nullable = true)



In [9]:
# retrieve all airports known to the FlightAware API
# comment added to prevent api calls

# get_all_airports()

In [10]:
# load and inspect the retrieved airport codes

all_airports_df = spark.read.json("all_airports.json")
all_airports_df.printSchema()

root
 |-- AllAirportsResult: struct (nullable = true)
 |    |-- data: array (nullable = true)
 |    |    |-- element: string (containsNull = true)



In [11]:
# flatten the airport codes data frame

all_airports_table = all_airports_df.select(explode("AllAirportsResult.data"))
print(all_airports_table.count())
all_airports_table.printSchema()

27296
root
 |-- col: string (nullable = true)



In [12]:
# create a list from the flattened airport code data frame

all_airports = [x.col for x in all_airports_table.collect()]

In [13]:
# retrieve and count all codes from large airports with non-empty gps_code

large_airports = [row["gps_code"] for row in ac_df.filter(ac_df["type"] == "large_airport").collect()]
large_airports = list(filter(lambda x: x != None, large_airports))
print(len(large_airports))

601


In [14]:
# retrieve and count all codes from medium airports with non-empty gps_code

medium_airports = [row["gps_code"] for row in ac_df.filter(ac_df["type"] == "medium_airport").collect()]
medium_airports = list(filter(lambda x: x != None, medium_airports))
print(len(medium_airports))

4279


In [15]:
# retrieve and count all codes from small airports with non-empty gps_code

small_airports = [row["gps_code"] for row in ac_df.filter(ac_df["type"] == "small_airport").collect()]
small_airports = list(filter(lambda x: x != None, small_airports))
print(len(small_airports))

25969


In [16]:
# retrieve actual flight data
# comments added to prevent api calls

# set_max_result_size(15000)

# specify for which airports the traffic data should be retrieved and stored in JSON files
airports = medium_airports
# increase the index for consecutive runs
idx = 2

for i in range(len(airports)):
    # don't send API requests when airport is not known to FlightAware
    if (airports[i] not in all_airports):
        continue
    arrived_file = Path("Arrived/{}-{}.json".format(airports[i], idx))
    departed_file = Path("Departed/{}-{}.json".format(airports[i], idx))
    enroute_file = Path("Enroute/{}-{}.json".format(airports[i], idx))
    if i % 100 == 0:
        print(i)
    # don't send a request if the resulting file already exists
    if not departed_file.is_file():
        pass
        # get_for_airport("Departed", airports[i], idx)
    if not arrived_file.is_file():
        pass
        # get_for_airport("Arrived", airports[i], idx)
    if not enroute_file.is_file():
        pass
        # get_for_airport("Enroute", airports[i], idx)

0
100
200
300
400
500
600
700
900
1100
1200
1300
1400
1500
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3600
3700
3800
3900
4000
4100
4200


In [17]:
deps_df = spark.read.json("s3a://eseidinger-data-engineering/flight_data/Departed/*.json")
# deps_df = spark.read.json("Departed/*.json")

In [18]:
print(deps_df.count())
deps_df.printSchema()

16487
root
 |-- DepartedResult: struct (nullable = true)
 |    |-- departures: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- actualarrivaltime: long (nullable = true)
 |    |    |    |-- actualdeparturetime: long (nullable = true)
 |    |    |    |-- aircrafttype: string (nullable = true)
 |    |    |    |-- destination: string (nullable = true)
 |    |    |    |-- destinationCity: string (nullable = true)
 |    |    |    |-- destinationName: string (nullable = true)
 |    |    |    |-- estimatedarrivaltime: long (nullable = true)
 |    |    |    |-- ident: string (nullable = true)
 |    |    |    |-- origin: string (nullable = true)
 |    |    |    |-- originCity: string (nullable = true)
 |    |    |    |-- originName: string (nullable = true)
 |    |-- next_offset: long (nullable = true)



In [19]:
arrs_df = spark.read.json("s3a://eseidinger-data-engineering/flight_data/Arrived/*.json")
# arrs_df = spark.read.json("Arrived/*.json")

In [20]:
print(arrs_df.count())
arrs_df.printSchema()

16487
root
 |-- ArrivedResult: struct (nullable = true)
 |    |-- arrivals: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- actualarrivaltime: long (nullable = true)
 |    |    |    |-- actualdeparturetime: long (nullable = true)
 |    |    |    |-- aircrafttype: string (nullable = true)
 |    |    |    |-- destination: string (nullable = true)
 |    |    |    |-- destinationCity: string (nullable = true)
 |    |    |    |-- destinationName: string (nullable = true)
 |    |    |    |-- ident: string (nullable = true)
 |    |    |    |-- origin: string (nullable = true)
 |    |    |    |-- originCity: string (nullable = true)
 |    |    |    |-- originName: string (nullable = true)
 |    |-- next_offset: long (nullable = true)



In [21]:
enr_df = spark.read.json("s3a://eseidinger-data-engineering/flight_data/Enroute/*.json")
# enr_df = spark.read.json("Enroute/*.json")

In [22]:
print(enr_df.count())
enr_df.printSchema()

20251
root
 |-- EnrouteResult: struct (nullable = true)
 |    |-- enroute: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- actualdeparturetime: long (nullable = true)
 |    |    |    |-- aircrafttype: string (nullable = true)
 |    |    |    |-- destination: string (nullable = true)
 |    |    |    |-- destinationCity: string (nullable = true)
 |    |    |    |-- destinationName: string (nullable = true)
 |    |    |    |-- estimatedarrivaltime: long (nullable = true)
 |    |    |    |-- filed_departuretime: long (nullable = true)
 |    |    |    |-- ident: string (nullable = true)
 |    |    |    |-- origin: string (nullable = true)
 |    |    |    |-- originCity: string (nullable = true)
 |    |    |    |-- originName: string (nullable = true)
 |    |-- next_offset: long (nullable = true)



### Step 2: Explore and Assess the Data
#### Explore the Data 
Some rows in the airport data contain no gps_code which is needed to associate with the airport in the flight data.
The flight data is nested and needs to be flattend. The flight data contains duplicates.

#### Cleaning Steps
See comments in code below for explanation of the cleaning steps.

In [23]:
# Flatten departure data frame

deps_table = deps_df.select(explode("DepartedResult.departures"))
deps_table = deps_table.select("col.*")
print(deps_table.count())
deps_table.printSchema()

255622
root
 |-- actualarrivaltime: long (nullable = true)
 |-- actualdeparturetime: long (nullable = true)
 |-- aircrafttype: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destinationCity: string (nullable = true)
 |-- destinationName: string (nullable = true)
 |-- estimatedarrivaltime: long (nullable = true)
 |-- ident: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- originCity: string (nullable = true)
 |-- originName: string (nullable = true)



In [24]:
# flatten arrivals data frame

arrs_table = arrs_df.select(explode("ArrivedResult.arrivals"))
arrs_table = arrs_table.select("col.*")
print(arrs_table.count())
arrs_table.printSchema()

255784
root
 |-- actualarrivaltime: long (nullable = true)
 |-- actualdeparturetime: long (nullable = true)
 |-- aircrafttype: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destinationCity: string (nullable = true)
 |-- destinationName: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- originCity: string (nullable = true)
 |-- originName: string (nullable = true)



In [25]:
# flatten enroute data frame

enr_table = enr_df.select(explode("EnrouteResult.enroute"))
enr_table = enr_table.select("col.*")
print(enr_table.count())
enr_table.printSchema()

515577
root
 |-- actualdeparturetime: long (nullable = true)
 |-- aircrafttype: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destinationCity: string (nullable = true)
 |-- destinationName: string (nullable = true)
 |-- estimatedarrivaltime: long (nullable = true)
 |-- filed_departuretime: long (nullable = true)
 |-- ident: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- originCity: string (nullable = true)
 |-- originName: string (nullable = true)



In [26]:
# filter invalid airport data

ac_df_valid = ac_df.dropna(how="any", subset=["gps_code"])
ac_df_valid.count()

41030

In [27]:
deps_table = deps_table.drop_duplicates()
deps_table.count()

197698

In [28]:
arrs_table = arrs_table.drop_duplicates()
arrs_table.count()

186248

In [29]:
enr_table = enr_table.drop_duplicates()
enr_table.count()

358836

### Step 3: Define the Data Model
#### 3.1 Conceptual Data Model

![Data Model](model.png)

I chose a star schema with flights as facts table in the center and airports and time as dimension tables. This schema allows me to make queries about the frequency of flights from an to certain airports and when these flights occur.

#### 3.2 Mapping Out Data Pipelines

* Departure, arrival and enroute flight data has to be downlaoded from the FlightAware API and stored in intermediary JSON files (Step 1)
* Airport data needs to be loaded from the given csv file (Step 1)
* The above data needs to be loaded in Spark data frames for staging (Step 1)
* Data cleaning has to be performed (Step 2)
* The time data frame can be extracted from the departure, arrival and enroute flight staging data frames (Step 4)
* Each of the departure, arrival and enroute flight staging data frames has to be joined with the airport staging data frame to extract airport data (Step 4)
* The flight data frame can be extracted from the departure, arrival and enroute flight staging data frames (Step 4)

### Step 4: Run Pipelines to Model the Data 
#### 4.1 Create the data model

In [30]:
# build the time table
time_table = deps_table.select("actualarrivaltime").withColumnRenamed("actualarrivaltime", "timestamp")
time_table = time_table.union(deps_table.select("actualdeparturetime").withColumnRenamed("actualdeparturetime", "timestamp"))
time_table = time_table.union(deps_table.select("actualdeparturetime").withColumnRenamed("actualdeparturetime", "timestamp"))
time_table = time_table.union(arrs_table.select("actualarrivaltime").withColumnRenamed("actualarrivaltime", "timestamp"))
time_table = time_table.union(arrs_table.select("actualdeparturetime").withColumnRenamed("actualdeparturetime", "timestamp"))
time_table = time_table.union(enr_table.select("estimatedarrivaltime").withColumnRenamed("estimatedarrivaltime", "timestamp"))
time_table = time_table.union(enr_table.select("filed_departuretime").withColumnRenamed("filed_departuretime", "timestamp"))

time_table.drop_duplicates()

get_datetime = udf(datetime.fromtimestamp, TimestampType())
time_table = time_table.withColumn("datetime", get_datetime(col("timestamp"))) \
    .withColumn("year", year("datetime")) \
    .withColumn("month", month("datetime")) \
    .withColumn("dayofmonth", dayofmonth("datetime")) \
    .withColumn("hour", hour("datetime"))

print(time_table.count())
time_table.printSchema()

1683262
root
 |-- timestamp: long (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- dayofmonth: integer (nullable = true)
 |-- hour: integer (nullable = true)



In [32]:
time_table.write.parquet("output/time_out")

In [33]:
# build the airport table
get_lon = udf(lambda x: float(x.split(",")[0]), FloatType())
get_lat = udf(lambda x: float(x.split(",")[1]), FloatType())
ac_df_valid = ac_df_valid.withColumn("latitude", get_lat(col("coordinates"))).withColumn("longitude", get_lon(col("coordinates")))

airport_table = ac_df_valid.withColumnRenamed("ident", "id") \
    .join(deps_table, [ac_df_valid.gps_code == deps_table.origin]) \
    .select("origin", "originName", "originCity", "continent", "iso_country", "latitude", "longitude") \
    .withColumnRenamed("origin", "id") \
    .withColumnRenamed("originName", "name") \
    .withColumnRenamed("originCity", "city") \
    .withColumnRenamed("iso_country", "country")

airport_table = airport_table.union(ac_df_valid.withColumnRenamed("ident", "id") \
    .join(deps_table, [ac_df_valid.gps_code == deps_table.destination]) \
    .select("destination", "destinationName", "destinationCity", "continent", "iso_country", "latitude", "longitude") \
    .withColumnRenamed("destination", "id") \
    .withColumnRenamed("destinationName", "name") \
    .withColumnRenamed("destinationCity", "city") \
    .withColumnRenamed("iso_country", "country"))

airport_table = airport_table.union(ac_df_valid.withColumnRenamed("ident", "id") \
    .join(arrs_table, [ac_df_valid.gps_code == arrs_table.origin]) \
    .select("origin", "originName", "originCity", "continent", "iso_country", "latitude", "longitude") \
    .withColumnRenamed("origin", "id") \
    .withColumnRenamed("originName", "name") \
    .withColumnRenamed("originCity", "city") \
    .withColumnRenamed("iso_country", "country"))

airport_table = airport_table.union(ac_df_valid.withColumnRenamed("ident", "id") \
    .join(arrs_table, [ac_df_valid.gps_code == arrs_table.destination]) \
    .select("destination", "destinationName", "destinationCity", "continent", "iso_country", "latitude", "longitude") \
    .withColumnRenamed("destination", "id") \
    .withColumnRenamed("destinationName", "name") \
    .withColumnRenamed("destinationCity", "city") \
    .withColumnRenamed("iso_country", "country"))

airport_table = airport_table.union(ac_df_valid.withColumnRenamed("ident", "id") \
    .join(enr_table, [ac_df_valid.gps_code == enr_table.origin]) \
    .select("origin", "originName", "originCity", "continent", "iso_country", "latitude", "longitude") \
    .withColumnRenamed("origin", "id") \
    .withColumnRenamed("originName", "name") \
    .withColumnRenamed("originCity", "city") \
    .withColumnRenamed("iso_country", "country"))

airport_table = airport_table.union(ac_df_valid.withColumnRenamed("ident", "id") \
    .join(enr_table, [ac_df_valid.gps_code == enr_table.destination]) \
    .select("destination", "destinationName", "destinationCity", "continent", "iso_country", "latitude", "longitude") \
    .withColumnRenamed("destination", "id") \
    .withColumnRenamed("destinationName", "name") \
    .withColumnRenamed("destinationCity", "city") \
    .withColumnRenamed("iso_country", "country"))

airport_table = airport_table.drop_duplicates()

print(airport_table.count())
airport_table.printSchema()

6176
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: float (nullable = true)
 |-- longitude: float (nullable = true)



In [34]:
airport_table.write.parquet("output/airport_out")

In [35]:
# build the flight table

flight_table = deps_table.select("ident", "aircrafttype", "origin", "destination", "actualdeparturetime", "actualarrivaltime") \
    .withColumnRenamed("ident", "id") \
    .withColumnRenamed("aircrafttype", "aircraft_type") \
    .withColumnRenamed("actualdeparturetime", "departure_time") \
    .withColumnRenamed("actualarrivaltime", "arrival_time")

flight_table = flight_table.union(arrs_table.select("ident", "aircrafttype", "origin", "destination", "actualdeparturetime", "actualarrivaltime") \
    .withColumnRenamed("ident", "id") \
    .withColumnRenamed("aircrafttype", "aircraft_type") \
    .withColumnRenamed("actualdeparturetime", "departure_time") \
    .withColumnRenamed("actualarrivaltime", "arrival_time"))

flight_table = flight_table.union(enr_table.select("ident", "aircrafttype", "origin", "destination", "filed_departuretime", "estimatedarrivaltime") \
    .withColumnRenamed("ident", "id") \
    .withColumnRenamed("aircrafttype", "aircraft_type") \
    .withColumnRenamed("filed_departuretime", "departure_time") \
    .withColumnRenamed("estimatedarrivaltime", "arrival_time"))

flight_table = flight_table.drop_duplicates()

print(flight_table.count())
flight_table.printSchema()

471724
root
 |-- id: string (nullable = true)
 |-- aircraft_type: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- departure_time: long (nullable = true)
 |-- arrival_time: long (nullable = true)



In [36]:
flight_table.write.parquet("output/flight_out")

#### 4.2 Data Quality Checks

In [37]:
# check for completenes

raw_flight_data_count = deps_table.count() + arrs_table.count() + enr_table.count()
flight_table_count = flight_table.count()

print(flight_table_count / raw_flight_data_count)

0.6350773174363407


63% of the raw flight data is in the final flight table. This is reasonable because flights included in the departures query results may also be included in the query results for enroute flights and in the query results for arrivals. Since duplicates are removed, this result is expected.

In [38]:
# checking the flight table for null values

flight_table.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in flight_table.columns]).show()

+---+-------------+------+-----------+--------------+------------+
| id|aircraft_type|origin|destination|departure_time|arrival_time|
+---+-------------+------+-----------+--------------+------------+
|  0|            0|     0|          0|             0|           0|
+---+-------------+------+-----------+--------------+------------+



As the above check shows, the flights data frame does not contain any null values.

#### 4.3 Data dictionary 

##### Flight data

All flight data was retrieved with the FlightAware API.

|data field     |description                                 |
|---------------|--------------------------------------------|
|id             |Designation of flight or tail number        |
|aircraft_type  |Type of aircraft used for the flight        |
|origin         |ICAO designation of the airport of origin   |
|destination    |ICAO designation of the destination airport |
|departure_time |Unix timestamp of the time of departure     |
|arrival_time   |Unix timestamp of the time of arrival       |

##### Airport data frame

Airport data was taken from the FlightAware API as well as from the provided csv file.

|data field |description
|-----------|------------------------------------|
|id         |ICAO designation of the airport     |
|name       |Name of the airport                 |
|city       |City the ariport belongs to         |
|continent  |Continent the airport belongs to    |
|country    |Country the airport belongs to      |
|latitude   |Latitude coordinate of the airport  |
|longitude  |Longitude coordinate of the airport |

##### Time data frame

|data field |description       |
|-----------|------------------|
|timestamp  |Unix timestamp    |
|hour       |Hour of the day   |
|day        |Day of the month  |
|month      |Month of the year |
|year       |Year              |

#### Step 5: Complete Project Write Up

I chose Spark for the data transformation because it can be run on a single machine as well as on a cluster of computers.

The flight data should be updated daily because queries of the FlightAware API contain only departures, arrivals and enroute flights for the last day. Differences in the amount of air traffic on different days can only be seen when performing the queries more often. On the other hand, the API is not for free so this can be rather expensive.

If the data was increased by 100x I would download the data onto a Spark / Hadoop cluster for staging and transformation. I would store the result in an S3 bucket.

To populate the dashboard on a daily basis I would create a DAG in Apache Airflow and schedule it to be execcuted at 7 am every morning.

If the database needed to be accessed by 100+ people, I would store the data in a Cassandra cluster.